# Introduction

This notebook demostrates a signal processing pipeline and allows for estimation of time needed for data postprocessing.


# Load Python modules


In [ ]:
import numpy as np

# Data loading
from lib.file_func import load_file
import json
import pandas as pd
from lib.file_func import get_instrument_type, get_sum_signal

# Plotting functions
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from matplotlib import pyplot as plt
from lib.image import gen_heatmap_image
import ipywidgets as widgets

# Force Plotly figure to be shown in jupyterlab
import plotly.io as pio

pio.renderers.default = "jupyterlab"

# Resolution function fitting
from lib.peak import fwhm_to_sigma
from hardware.tofwerk import TwFitResolution

# Mass calibration (TOF only)
from hardware.tofwerk.calibration import mz_calibrate

# Peak fitting
from scipy.signal import find_peaks, peak_widths, savgol_filter
from lib.peak import detect_peaks, gen_peak
from lmfit.models import (
    SplineModel,
    SkewedGaussianModel,
)
from scipy.stats import linregress
from scipy.optimize import curve_fit
from scipy.interpolate import CubicSpline

# Supress pandas future warnings
import warnings

warnings.simplefilter(action="ignore", category=FutureWarning)

# Matching analysis
from backend.api.controllers.matches_controller import compute_matches

# File conversion

File Converter is running in a server as a separate application, communicating with Mascope.
The application keeps track of changes in the chosen folder and if a new raw file appears, it is converted to xarr format.

Conversion of each file takes up to ~30 s.
Currently, the mz precision can be set in `RawStreamer` class, the default value is 4.

Precision justification from Oskari Kausiala:

> We might also think of it in terms of the mass accuracy. The unit for m/z error we typically use is the relative error in ppm. That is, dmz[ppm] = 1e6 \* dmz[absolute] / mz
>
> 3 decimal place precision at mass 100 would be: $1e6 \cdot \frac{0.001}{100} = 10$ ppm
>
> 4 decimal place precision at mass 100 would be: $1e6 \cdot \frac{0.0001}{100} = 1$ ppm
>
> The Orbitrap should be capable of mass precision of about 1 ppm, so quantizing the data to 10 ppm is probably not optimal. On the other hand, 5 decimal places and thus 0.1 ppm quantization might not give us much more information

TODO: link to PDF file containing the comparison of peak shapes at different precision.

Note: to run the application, run the command `poetry run file-converter --st Raw --source directory_to_be_monitored` in Power Shell from the "Mascope/backend" folder.


# Load files

Currently we have two types of data files: from Orbitrap mass spectrometer (MS) and TOF MS.
The expected structure of data is the following:

    |  path_to_main_folder
    |   |  instrument
    |   |   |  name_of_the_instrument (KLTOF1 or KORBI2, parsed from the data file name)
    |   |   |   |  date (YYYY.MM.DD, parsed from the data file name)
    |   |   |   |   |  zarr_data_folder (see below)

The input data is stored in [xarray](https://docs.xarray.dev/en/stable/index.html) multi-dimensional array format.
The averaged spectrum and mass-to-charge (mz) values are used for further processing.


In [ ]:
# Load data file
# data_path = "C:/mascope_data"
file_name = "KLTOF1_2023.11.09_13h31m50s"
f = load_file(file_name)
# Define the MS
instrument_type = get_instrument_type(file_name)
# Extract averaged spectrum and mz values
spec = f.signal.interpolate_na(dim="mz", method="linear").sum(dim="time")
mz = spec.mz.values
spec = spec.values
# Smooth spectrum for better fitting if needed
# spec = savgol_filter(
#     spec,
#     window_length=5,
#     polyorder=3
# )
# The range can be adjusted if needed
# spec = spec[mz>100]
# mz = mz[mz>100]

# Find peaks. Tune the threshold if there are no peaks/too many peaks detected.
# TODO any better way to set threshhold?
peak, _ = find_peaks(spec, height=np.percentile(spec, 99.0))
fwhm = peak_widths(spec, peak, rel_height=0.5)[0]
sig = np.array([fwhm_to_sigma(fw) for fw in fwhm])

# Plot result
fig = go.Figure(
    [
        go.Scatter(x=mz, y=spec, name="Raw counts"),
        go.Scatter(x=mz[peak], y=spec[peak], mode="markers", name="Detected peaks"),
    ]
)
fig.update_layout(
    height=300, width=800, margin=go.layout.Margin(l=30, r=30, b=30, t=30, pad=4)
)
print(f"{len(peak)} peaks were detected.")
fig.show()

# Peak shape estimation

Sometimes the peaks in mass-spectra are asssumed to have symmetric Gaussian peak-shape.
In practice, peak shapes vary.

There are several mathematical approaches descibing the peak shape, however, they are not universal.
Here we adopt the method proposed by [DeCarlo et al, 2016](https://pubs.acs.org/doi/10.1021/ac061249n).
This method incorporates determined peak widths from experimental data to define peak width vs mz dependence and resolution function to enhance the accuracy of peak fitting in mass spectrometry data.

The algorithm works as follows:

- We go through the list of peaks found above and look at a window mz $_i$ $\pm$ dmz.
- The counts in a window are normalized to 1, the mz scale is shifted to 0 $\pm$ dmz.
- The peaks having a too high baseline or the peak which are not the highest in a windows are skipped.
- We try to fit the spectrum in a window with the Gaussian peak-shape.
- If R-squared of the fit is lower than 0.95 (to far from Gaussian), the peak is dismissed.
- Counts and mz scales are refined based on a Gaussian fit position and height.
- Gaussian full width half maximum is stored for resolving power (resolution function) calculations.
- The peak width is normalized to sigma=1 (mz scale is divided by Gaussian fit sigma).
- Normalized and refined peak-shape is stored.
- Stored peak-shapes are used for calculating a median peak shape.

The median peak shape will be used later for a final deconvolution of a mass spectrum.


In [ ]:
def fit_gaussian(x, y):
    """Fit the spectrum range with a skewed Gaussian peak-shape using lmfit

    :param x: mz scale
    :type x: array-like
    :param y: counts
    :type y: array-like
    :return: ModelResult (see lmfit doc)
    :rtype: ModelResult
    """
    # Initialize fitting parameters for the main peak
    model = SkewedGaussianModel(prefix="p_")
    params = model.make_params()
    if instrument_type == "tof":
        # Fitting parameters for the background
        knot_xvals = np.array([-dmz, -dmz / 2, -dmz / 3, dmz / 3, dmz / 2, dmz])
        bkg = SplineModel(prefix="bkg_", xknots=knot_xvals)
        params.update(bkg.guess(y, x))
        # Total model
        model = model + bkg

    params.add("p_amplitude", value=1, max=1.1)
    params.add("p_center", value=0)
    params.add("p_sigma", value=0.01)
    params.add("p_gamma", value=0.1)

    # Perform fitting
    fit = model.fit(y, params, x=x)
    return fit


def calculate_fwhm(x, y):
    # Find the maximum count and its index
    max_index = np.argmax(y)
    max_count = y[max_index]

    # Find the half maximum count
    half_max_count = max_count / 2

    # Find the indices where the counts are closest to the half maximum
    left_index = np.argmin(np.abs(y[:max_index] - half_max_count))
    right_index = np.argmin(np.abs(y[max_index:] - half_max_count)) + max_index

    # Calculate the FWHM
    fwhm = x[right_index] - x[left_index]

    return fwhm


def calculate_center_of_mass(x, y):
    # Calculate the center of mass
    center_of_mass = np.sum(x * y) / np.sum(y)
    return center_of_mass

In [ ]:
# Expected half-width band presumably containing one peak
# Decrease dmz if you get too many peaks in one band
# Increase dmz if the peak shape is distorted/cut
dmz = 0.1  # TODO find way to automatically estimate optimal dmz value
r_squared_threshold = 0.9

peak_traces = []
peak_norm_traces = []

p_x = np.linspace(-10, 10, 101)
p_ys = []
p_mzs = []
p_fwhms = []
p_centers = []

for i, p in enumerate(peak):
    p_height = spec[p]
    # Select a narrow region (peak center +/- dmz) of the spectrum around the peak
    p_mz_center = mz[p]
    p_mz0 = p_mz_center - dmz
    p_mz1 = p_mz_center + dmz
    # Find indices
    p_i0 = np.argmin(np.abs(mz - p_mz0))  # left border
    p_i1 = np.argmin(np.abs(mz - p_mz1))  # right border
    p_i = p - p_i0  # center
    # Peak region
    p_mz = mz[p_i0:p_i1]
    p_spec = spec[p_i0:p_i1]

    if np.max(p_spec) > p_height:
        # 'p' is not the biggest peak in range, dismiss
        continue

    # Normalize peak region: mz around 0 and spec to range [0, 1]
    p_mz_norm = p_mz - p_mz_center
    p_spec_norm = p_spec / p_height
    p_spec_norm -= p_spec_norm.min()

    peak_traces.append(go.Scatter(x=p_mz_norm, y=p_spec_norm))
    # Fit peak in the region
    fit = fit_gaussian(p_mz_norm, p_spec_norm)
    p_spec_norm_fit = fit.eval_components()["p_"]

    if fit.rsquared < r_squared_threshold:
        # fitting error to large, dismiss
        continue

    # Remove junk peaks arond main one
    mask = np.where(fit.best_fit < 1e-9)
    p_spec_norm[mask] = 0

    # Get peak top location

    if instrument_type == "tof":
        # Remove baseline if TOF
        p_spec_norm -= fit.eval_components()["bkg_"]
        p_spec_norm[np.where(p_spec_norm < 0)] = 0

    top_y = np.max(p_spec_norm_fit)
    top_x = calculate_center_of_mass(p_mz_norm, p_spec_norm_fit)

    # Refine peak position and height
    p_mz_norm -= top_x
    p_spec_norm /= top_y

    # if np.abs(p_mz_norm[np.argmin(np.abs(p_spec_norm - np.max(p_spec_norm)))]) > 5e-3:
    #     continue

    # Get and store Gaussian peak sigma and width
    try:
        p_fwhm = calculate_fwhm(p_mz_norm, p_spec_norm_fit)
    except Exception:
        continue

    # p_sigma = fit.params["p_sigma"].value
    p_sigma = p_fwhm / (2 * np.sqrt(2 * np.log(2)))
    # Scale peak to width sigma=1
    p_mz_norm /= p_sigma
    # Interpolate the normalized (both width and height) peak into predefined domain "p_x"
    p_y = np.interp(p_x, p_mz_norm, p_spec_norm, left=0, right=0)
    if np.all(np.isnan(p_y)):
        continue
    # Store peak positions, widths, and refined peak shape
    p_mzs.append(p_mz_center)
    p_centers.append(top_x)
    p_fwhms.append(p_fwhm)
    p_ys.append(p_y)
    peak_norm_traces.append(go.Scatter(x=p_x, y=p_y, name=f"peak {i}"))

# Clean shifted outliers
p_centers = np.array(p_centers)
non_outlier_indx = np.where(p_centers - np.median(p_centers) < 1e-3)[0]
peak_norm_traces = [peak_norm_traces[i] for i in non_outlier_indx]
p_fwhms = [p_fwhms[i] for i in non_outlier_indx]
p_ys = [p_ys[i] for i in non_outlier_indx]
p_mzs = [p_mzs[i] for i in non_outlier_indx]

# Calculate median peak shape
p_median = np.median(np.array([p_y for p_y in p_ys]), axis=0)
# Check if p_median is empty
if p_median.all() == np.nan:
    raise Exception(
        """p_median is empty
        Probably fitting error threshold is too strict"""
    )
# Add to the figure
peak_norm_traces.append(
    go.Scatter(x=p_x, y=p_median, line={"color": "red", "width": 3}, name="median")
)
# Plot the output
fig = go.FigureWidget(
    peak_norm_traces,
    {"title": f"Median peak shape of {len(peak_norm_traces)-1} peaks"},
)
fig.update_layout(
    height=300, width=800, margin=go.layout.Margin(l=30, r=30, b=30, t=30, pad=4)
)


def toggle_trace_visibility(trace, points, selector):
    """Updates the median peak shape based on shapes present in the figure"""
    if not points.point_inds:
        return
    trace.visible = "legendonly"

    selected_traces = [
        trace.y
        for trace in fig.data
        if ("peak" in trace.name and trace.visible != "legendonly")
    ]

    fig.update_layout(
        title=f"Median peak shape of {len(selected_traces)} peaks",
        height=300,
        width=800,
        margin=go.layout.Margin(l=30, r=30, b=30, t=30, pad=4),
    )
    fig.update_traces(
        {"y": np.median(np.array(selected_traces), axis=0)},
        selector=lambda trace: trace.name == "median",
    )


print("Click on distorted peaks to remove them from peak shape estimation.")
[trace.on_click(toggle_trace_visibility) for trace in fig.data]
fig

Save peak shape and check how good the mean shape align with peaks along mz scale, R-squared used as error estimation.


In [ ]:
# Save peak
peak_shape = {"x": p_x, "y": p_median}
# To JSON file if needed
# with open("C:/Users/alvai/Desktop/peak_shape.json", "w") as f:
#     json.dump({"x": list(p_x), "y": list(p_median)}, f)
p_ys = np.array(p_ys)
# Calculate residuals for normalized peaks vs median peak shape
norm_residuals = p_ys - peak_shape["y"]
## calculate R-squared vs mz
r_sq_norm = 1 - (norm_residuals**2).sum(axis=1) / ((p_ys - p_ys.mean()) ** 2).sum(
    axis=1
)
# Plot error vs mz
err_fig = plt.figure(figsize=(7, 1.5))
plt.scatter(p_mzs, r_sq_norm)
plt.xlabel("mz")
plt.ylabel(r"R$^2$")

## Resolution function

The resolving power or resolution function of MS depends on mass-charge and defines the width of peaks we observe in spectrum.
The knowledge of resolution (peak width) vs mz dependence is crucial for the spectrum deconvolution because it allows for resolving cases with overlapping or noisy peaks.

### TOF

TOF [documentation](https://soft.tofwerk.com/TofDaq_1.99r759_API_20180912.zip) proposes the following empirical equation for fitting mz/resolution pairs obtained from experimental data:

$$R(mz) = R_0 - \frac{R_0}{1 + e^{\frac{mz - mz_0}{dmz}}}$$

where $R(mz)$ - resolution function, $R_0$ - ultimate resolving power, $mz_0$ - mass, where $R(mz) = 0.5\cdot R_0$, $dmz$ - slope parameter.

### Orbitrap

In general, the experimental mass resolution for Orbitrap is ([Perry et al, 2008](https://doi.org/10.1002/mas.20186)):

$$R(mz) = \frac{1}{2 \Delta \omega _{50\%}} \sqrt{\frac{k \cdot q}{mz}}$$

where $q$ is a charge of an ion inside the trap, $k$ is the axial restoring force, $\Delta \omega _{50\%}$ is the difference in ion frequency at the FWHM.
From the equation we see, that $R \propto \frac{1}{\sqrt{mz}}$.
Thus, in case of Orbitrap, we will fit experimental mz/resolution pairs with a curve $R(mz) = \frac{a}{\sqrt{mz}}$, where $a$ is a fitting parameter.


In [ ]:
def R_tof(mz):
    return R0 - R0 / (1 + np.exp((mz - mz0) / dmz))


def R_orb(mz):
    return R_orb_coef / np.sqrt(mz)

Support functions


In [ ]:
def get_line(x, a, b):
    x = np.array(x)
    return a * x + b


def get_polynome(x, a, b, c):
    x = np.array(x)
    return a * x**2 + b * x + c


def get_inverse_sqrt(x, a):
    x = np.array(x)
    return a / np.sqrt(x)

Fitting of the resolution function


In [ ]:
# Get the fitted line depending on the instrument
if "tof" in file_name.lower():
    regres = linregress(p_mzs, p_fwhms)
    p_fwhms_line = get_line(p_mzs, regres.slope, regres.intercept)
if "orbi" in file_name.lower():
    coefs = np.polyfit(p_mzs, p_fwhms, 2)
    p_fwhms_line = get_polynome(p_mzs, *coefs)

# Points higher than ndev * std_dev will be later filtered out
ndev = 1
# Get residuals and standard deviation
residuals = p_fwhms - p_fwhms_line
std_dev = np.std(residuals)
is_outlier = (residuals > ndev * std_dev) | (residuals < -ndev * std_dev)

# Plot figure to see outliyers
fig, (ax_res, ax_fwhm) = plt.subplots(nrows=1, ncols=2, figsize=(12, 3))
ax_fwhm.plot(p_mzs, p_fwhms, label="True FWHM")
ax_fwhm.plot(p_mzs, p_fwhms_line, linestyle="--", color="red", label="Approximation")
ax_fwhm.fill_between(
    p_mzs,
    p_fwhms_line + ndev * std_dev,
    p_fwhms_line - ndev * std_dev,
    alpha=0.2,
    label="Non-outliers",
)
ax_fwhm.set(ylabel="FWHM", xlabel="mz")
ax_fwhm.legend()

# Remove outliers
p_fwhms_filt = np.array(p_fwhms, dtype=np.double)[~is_outlier]
mass = np.array(p_mzs, dtype=np.double)[~is_outlier]
resolution = mass / p_fwhms_filt

# the intrument is TOF
if "tof" in file_name.lower():
    # Prepare arguments for TwFitResolution
    nbrPoints = len(mass)
    # Initial guesses for fitting function
    R0 = np.array([np.max(resolution)], dtype=np.double)
    mz0 = np.array(
        [mass[(np.abs(resolution - resolution.mean())).argmin()]], dtype=np.double
    )
    dmz = np.array([np.percentile(mass, 70) - np.percentile(mass, 30)], dtype=np.double)
    # Get fitted parameters for TOF resolution function
    ret = TwFitResolution(nbrPoints, mass, resolution, R0, mz0, dmz)
    R0, mz0, dmz = R0[0], mz0[0], dmz[0]

    print(f"TOF, code {ret}, R0={R0:.3f}, m0={mz0:.3f}, dm={dmz:.3f}")

    resolution_function = R_tof
# the intrument is Orbitrap
if "orbi" in file_name.lower():
    # Fit resolution with the inverse square root
    try:
        popt, pcov = curve_fit(get_inverse_sqrt, mass, resolution)
        R_orb_coef = popt[0]
    except ValueError:
        print("No points available to fit resolution function")
        R_orb_coef = 1.725e6
    resolution_function = R_orb

    print(f"Orbi, a={R_orb_coef:.3f}")

# Plot fit
ax_res.scatter(
    p_mzs, np.array(p_mzs) / np.array(p_fwhms), color="grey", label="Ommited pairs"
)
ax_res.scatter(mass, resolution, color="black", label="Used mass/resolution pairs")

mass_range = np.linspace(min(p_mzs), max(p_mzs), 100)
ax_res.plot(
    mass_range, resolution_function(mass_range), "r", label="Fitted resolution function"
)
ax_res.set(
    ylabel="Resolution",
    xlabel="mz",
    title="Resolution function",
)
ax_res.legend()

# Mass calibration (TOF only)

In time-of-flight mass spectrometry, ions are identified based on the time it takes them to travel a set distance.
Lighter ions move faster, reaching the detector quicker than heavier ones.
Flight times are then converted to mass-charge using calibration curve.

Determintaion of caibrations curve is an important step for accurate mass determintaion.
Even small changes in the acceleration voltage or flight path length (for example, due to temperature change) can alter ion velocities.

`TwMassCalibrate` function (see TOF [documentation](https://soft.tofwerk.com/TofDaq_1.99r759_API_20180912.zip)) is used to fit coefficients for calibration curve.
After that the mz scale of the sample file is updated.


In [ ]:
# TEMP had to isolate this function as well
from lib.file_func import (
    get_zarr_var_shape,
    load_coord,
    update_props,
    update_zarr_array_coord,
)
from hardware.tofwerk.lib.TwTool import TwTof2Mass
from zarr.errors import PathNotFoundError


def remove_duplicate_mz_values(mz):
    # Sometimes TOF signal mz coordinate contains multiple zeros at the beginning
    # This may cause duplicate coordinate value error in some functions
    # This function fixes the coordinate vector by setting arbitrary small values for
    # the zero coordinates
    mz_unique = mz
    mz_below_10_mask = mz < 10
    if (np.diff(mz[mz_below_10_mask]) == 0).any():
        mz_below_10_maxi = mz_below_10_mask.sum()
        mz_unique[mz_below_10_mask] = np.linspace(
            0, mz[mz_below_10_maxi], mz_below_10_maxi, endpoint=False
        )
    return mz_unique


def signal_mz_calibration_update(fit, filename):
    mode = fit["mode"]
    par = fit["par"]
    # Calculate new mz axis
    nbr_samples = get_zarr_var_shape(filename, "signal")[0]
    par = np.array(par, dtype=np.double)
    new_mz = np.array([TwTof2Mass(tof, mode, par) for tof in range(nbr_samples)])
    new_mz = remove_duplicate_mz_values(new_mz)
    new_range = [new_mz[0], new_mz[-1]]

    # Update zarr file coordinates and props
    print("Calibrating file: %s" % filename)
    if nbr_samples != get_zarr_var_shape(filename, "signal")[0]:
        raise Exception("Number of TOF samples does not match")
    update_props(filename, {"range": new_range, "mz_calibration": fit})
    # Write new mz coordinates to zarr file
    update_zarr_array_coord(filename, "signal", "mz", new_mz)
    try:
        update_zarr_array_coord(filename, "sum_signal", "mz", new_mz)
    except PathNotFoundError:
        pass
    try:
        peak_tofs = load_coord(filename, "peak_areas", "tof")
        new_peak_mzs = new_mz[peak_tofs.astype(int)]
        update_zarr_array_coord(filename, "peak_areas", "mz", new_peak_mzs)
        update_zarr_array_coord(filename, "peak_heights", "mz", new_peak_mzs)
    except PathNotFoundError:
        pass
    return new_mz

In [ ]:
if instrument_type == "tof":
    # Get reference peak positions
    # Curently taken from Mascope DB
    reference_df = pd.DataFrame(
        columns=["target_isotope_id", "target_ion_id", "mz", "relative_abundance"]
    )
    for _ in ("CH2BrO2-", "Br3-", "Br2-"):
        with open(f"C:\\mascope_data\\mass_calibration\\{_}.json") as file:
            reference_df = pd.concat(
                [reference_df, pd.DataFrame(json.load(file)["data"])], ignore_index=True
            )
    # Match reference peaks with peaks we have in a sample
    match_reference_df = await compute_matches(
        file_name,
        reference_df,
        instrument_functions=(peak_shape, resolution_function),
        min_isotope_abundance=0.2,
    )
    # Get mass calibration coefficients
    mass_calib, stats = mz_calibrate(
        peak_tof=match_reference_df.sample_peak_tof,
        peak_mz=match_reference_df.sample_peak_mz,
        exact_mz=match_reference_df.mz,
    )
    # Update mz scale
    signal_mz_calibration_update(mass_calib, file_name)

# Peak fitting

Knowing median peak shape and resolution function, we can start the fitting.
The procedure is the following:

1. The spectrum is being split into frames of widths mz $\pm$ dmz.
2. The residuals are calculated.
3. The script finds the highest data point in the frame.
4. A peak of known shape and width is fitted to this data point, saved, then subtracted from the signal in the frame.
5. The residuals are calculated and compared with previous value.
   Steps 2-4 are repeated until the difference in residuals doesn't exceed threshold or the maximum amount of peaks we expect in the window.
   For now, the max number of peaks and threshold are chosen arbitrary.


In [ ]:
# Assign peak fitting threshold depending on the instrument type
# Correct intrument type unsured by get_instrument_type
if instrument_type == "orbi":
    threshold = 0.8
if instrument_type == "tof":
    threshold = 0.9
sample_file_data, all_peak_mzs = await detect_peaks(
    file_name,
    (peak_shape, resolution_function),
    threshold,
    # u_list=np.arange(200, 300, 1),  # for testing
    max_n_peaks=5,
    if_exists="replace",
    dmz=0.5,
    return_peak_mzs=True,
    instrument_type=instrument_type,
)

# Check fitting results

The fitting results were saved in the xarray data file.
Now we can import the file and visualize peaks.
To restore the peaks we need to extract peak positions and heights and compute peak widths based on the resolution function.

**Warning**: plotting the whole range takes a lot of time.


In [ ]:
f = load_file(file_name)
fit_peak_hei = f.peak_heights.dropna(dim="mz", how="all").sum(dim="time")

In [ ]:
def rescale_mz(mz_min, mz_max, points_per_peak):
    """re-calculate mz grid mz grid"""
    mz_grid = []
    mz = mz_min
    while mz < mz_max:
        R_orb = resolution_function(mz)
        fwhm = mz / R_orb
        step = fwhm / points_per_peak
        mz_grid.append(mz + step)
        mz += step
    return np.array(mz_grid, dtype=np.float32)

In [ ]:
def update_figure(event):
    """Handles figure update upon changes in input fields of widgets"""
    # Read new value
    user_mz_value = float(event.new)
    # Find the nearest from list_of_peaks
    closest_ind = np.argmin(np.abs(win_mid - user_mz_value))

    # Get spectrum in the first window
    win_spec = f.sum_signal.sel(mz=filtered_mz_windows[closest_ind]).compute()
    win_spec = win_spec.interp(
        coords={
            "mz": rescale_mz(
                mz_min=win_spec.mz.min().values,
                mz_max=win_spec.mz.max().values,
                points_per_peak=10,
            )
        }
    ).dropna(dim="mz")
    # Get mz coordinate in a window
    win_mz = win_spec.mz.values
    # Convert spectrum values to numpy array
    win_spec = win_spec.values
    # Get list of peaks in mz window
    peaks_in_win = list_of_peaks[
        (list_of_peaks >= win_mz.min()) & (list_of_peaks <= win_mz.max())
    ]
    # Restore peak shape
    restored_peak = np.zeros_like(win_mz)
    for ppos in peaks_in_win:
        phei = fit_peak_hei.sel(mz=ppos, method="nearest").values
        pres = resolution_function(ppos)
        restored_peak += gen_peak(
            x=win_mz, ppos=ppos, phei=phei, pres=pres, ps=peak_shape
        )

    # Get residuals
    win_res = win_spec - restored_peak
    # Update fitting error
    SS_total = ((win_spec - win_spec.mean()) ** 2).sum()
    fit_r_squared = 1 - (win_res**2).sum() / SS_total
    # Update traces
    figure_widget.data[0].x = win_mz
    figure_widget.data[1].x = win_mz
    figure_widget.data[2].x = win_mz

    figure_widget.data[0].y = win_res
    figure_widget.data[1].y = restored_peak
    figure_widget.data[2].y = win_spec

    figure_widget.update_layout(
        uirevision=None, title=f"R-squared={fit_r_squared:.2f}", title_x=0.3
    )
    # Update values in slider and
    mz_slider.value = ppos
    mz_manual.value = ppos

Edit line 10 in the next cell if the window is too wide.


In [ ]:
all_peak_mzs.astype(np.float32)

In [ ]:
filtered_mz_windows

In [ ]:
# List of fitted peaks
# list_of_peaks = fit_peak_hei.mz.values
list_of_peaks = all_peak_mzs.astype(np.float32)
# Get full mz scale
mz_scale = f.mz.values
# Split the array into windows
mz_windows = []
current_segment = [mz_scale[0]]
for i in range(1, len(mz_scale)):
    if mz_scale[i] - current_segment[0] < 0.5:  # edit to change window width
        current_segment.append(mz_scale[i])
    else:
        mz_windows.append(current_segment)
        current_segment = [mz_scale[i]]
# Add left segment
mz_windows.append(current_segment)
# Find rows that contain any value from the list_of_peaks array
is_present = [np.any(np.isin(arr, list_of_peaks)) for arr in mz_windows]
# Filter mz_windows based on the mask
filtered_mz_windows = [arr for arr, present in zip(mz_windows, is_present) if present]
# Midpoints of windows
win_mid = np.array([np.round(np.median(i), 3) for i in filtered_mz_windows])
# Get spectrum in the first window
win_spec = f.sum_signal.sel(mz=filtered_mz_windows[0]).compute()
win_spec = win_spec.interp(
    coords={
        "mz": rescale_mz(
            mz_min=win_spec.mz.min().values,
            mz_max=win_spec.mz.max().values,
            points_per_peak=10,
        )
    }
).dropna(dim="mz")
# Get mz coordinate in a window
win_mz = win_spec.mz.values
# Convert spectrum values to numpy array
win_spec = win_spec.values
# Get list of peaks in mz window
peaks_in_win = list_of_peaks[
    (list_of_peaks >= win_mz.min()) & (list_of_peaks <= win_mz.max())
]
# Restore peak shape
restored_peak = np.zeros_like(win_mz)
for ppos in peaks_in_win:
    phei = fit_peak_hei.sel(mz=ppos, method="nearest").values
    pres = resolution_function(ppos)
    restored_peak += gen_peak(x=win_mz, ppos=ppos, phei=phei, pres=pres, ps=peak_shape)

# Get residuals
win_res = win_spec - restored_peak
# Get R-squared (fitting error)
SS_total = ((win_spec - win_spec.mean()) ** 2).sum()
fit_r_squared = 1 - (win_res**2).sum() / SS_total
# Create figure space
fig = make_subplots(rows=2, cols=1, row_heights=[0.2, 0.8], shared_xaxes=True)
# Create traces
residual_trace = go.Scatter(x=win_mz, y=win_res, name="Residuals")
fit_peak_trace = go.Scatter(x=win_mz, y=restored_peak, name="Fitted peak")
raw_peak_trace = go.Scatter(x=win_mz, y=win_spec, name="Raw spectrum")
# Add traces to figure
fig.add_trace(residual_trace, row=1, col=1)
fig.add_trace(fit_peak_trace, row=2, col=1)
fig.add_trace(raw_peak_trace, row=2, col=1)
# Update axes titles
fig.update_xaxes(title_text="mz", row=2, col=1)
fig.update_yaxes(title_text="Counts", row=2, col=1)
# Tune layout
fig.update_layout(
    width=600,
    height=400,
    margin=go.layout.Margin(l=30, r=30, b=30, t=30, pad=4),
    title=f"R-squared={fit_r_squared:.2f}",
    title_x=0.3,
)
## Create widgets
# Slider input
slider_description = widgets.HTML(value="Select a value for m/z:")
mz_slider = widgets.SelectionSlider(
    options=list_of_peaks,
    value=list_of_peaks[0],
    description="",
    disabled=False,
    continuous_update=False,
    orientation="horizontal",
    readout=True,
    layout=widgets.Layout(width="400px"),
)
# Text window input
manual_description = widgets.HTML(value="...or input manually")
mz_manual = widgets.FloatText(
    value=list_of_peaks[0],
    description="",
    layout=widgets.Layout(width="70px"),
    disabled=False,
)
figure_widget = go.FigureWidget(fig)
container = widgets.VBox(
    [slider_description, mz_slider, manual_description, mz_manual, figure_widget]
)
# Track changes in input widgets
mz_slider.observe(update_figure, names="value")
mz_manual.observe(update_figure, names="value")
# Show widgets
display(container)

# Match computation

Matching means comparison of found peaks with a list of target ions.
The target ion list is derived from target compounds and ionization mechanisms.

Detailed desciption is available in [Bitrix](https://karsa.bitrix24.com/bitrix/tools/disk/focus.php?objectId=146062&cmd=show&action=showObjectInGrid&ncc=1).


In [ ]:
# Load targets
with open("C:\\mascope_data\\targets\\20230510_stability.json") as file:
    target_isotopes = json.load(file)

# Convert to DataFrame
target_isotopes_df = pd.DataFrame(target_isotopes["data"])

match_isotope_df = await compute_matches(
    file_name,
    target_isotopes_df,
    instrument_functions=(peak_shape, resolution_function),
    min_isotope_abundance=0.2,
)

match_isotope_df.head()

# Notes

## Issues

### Ongoing

### Resolved

- **For some reason it plots more traces than in i have in peak_norm_traces**.
  My bad, I was confused by peak names.
- **During the attempt to estimate widths, I got "square" peaks with distorted shape**.
  This was resolved by increasing dmz value (approx. half band in which we can found a peak).
- **It seems like for calculation of the resolution functions all the peak shapes are used, even though outliers removed from the figure.**
  Outliers are now excluded from further calculations.
- **Imported Orbitrap data consists of triangle peaks. As Oskari explained, nothing can be done with that on this stage, the actual data acquisition should be performed with higher accuracy (up to the 4th number).**
  Switching to 4 decimal places caused a comb-like shape of peaks.
  That was due to tiny shifts in mz in spectra with time (see "TEST issue with decimal points in Orbitrap data conversion" section).
  If we average spectra instead of summing them we get a proper shape.
  We ended up precalculating the mz grid.
- **Black Formatter extension messes up the code in Jupyter cells (separates every line of code from the folowwing with atleast 2 \n) keeps deleting ;(allows to avoid printing function/method output after the cell) at the end of the line.**
  BlackFormatter was not turned on as a default code formatter.

## Thoughts

- Can Voigt peak shape describe the peaks better? UPD: not really, the shape is intistiguishible from Gaussian.
  However, the assymetric Gaussian peak-shape with a spline backgrounds works better for TOF spectra.
